In [1]:
import os
import glob
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

defects = ["defect_a", "defect_b", "defect_c", "defect_d"]
HEIGHT, WIDTH = 128, 800
BATCH_SIZE = 16
EPOCHS = 25
PATIENCE = 6              # stop if no improvement for this many epochs
QUICK = False             # True = tiny run just to check the code works
EMPTY_VALUE = "-1"        # what we write when NO defect is predicted (blank = "null" error on Kaggle)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if QUICK:
    EPOCHS = 2
print("device:", device)

device: cpu


In [2]:
root = "/kaggle/input/competitions/industrial-surface-defect-detection/public"
if not os.path.exists(root + "/train.csv"):
    root = "/kaggle/input/industrial-surface-defect-detection/public"

train_df = pd.read_csv(root + "/train.csv", dtype=str)
sample = pd.read_csv(root + "/sample_submission.csv", keep_default_na=False, dtype=str)
print("train:", train_df.shape, "\nsample submission:", sample.shape)

# one row per image, one column per defect (blank = no defect)
label_table = train_df.pivot(index="image_id", columns="defect_type", values="mask_rle")
label_table = label_table.reindex(columns=defects).fillna("").replace(EMPTY_VALUE, "")
print("train images:", len(label_table))
print("defects present:\n", (label_table != "").sum())

# CHECK THIS: how does the sample file write 'empty' masks?
print("most common mask_rle values in sample_submission:")
print(sample["mask_rle"].value_counts().head(3))

train: (45572, 5) 
sample submission: (12000, 4)
train images: 11393
defects present:
 defect_type
defect_a    1081
defect_b     313
defect_c    4100
defect_d    1707
dtype: int64
most common mask_rle values in sample_submission:
mask_rle
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [3]:
# find every image file (train images are in the train folder)
train_dir = root + "/train"
train_files = {}
for path in glob.glob(train_dir + "/**/*.*", recursive=True):
    name = os.path.splitext(os.path.basename(path))[0]
    train_files[name] = path

ids = list(label_table.index)
if QUICK:
    ids = ids[:300]
rles = label_table.loc[ids, defects].values      # table of RLE strings

# load all images + masks into memory (uint8 keeps it small)
images = np.zeros((len(ids), HEIGHT, WIDTH), dtype=np.uint8)
masks = np.zeros((len(ids), 4, HEIGHT, WIDTH), dtype=np.uint8)

for i in range(len(ids)):
    images[i] = np.array(Image.open(train_files[ids[i]]).convert("L"))
    for k in range(4):
        rle = rles[i, k]
        if rle.strip() != "":
            numbers = np.array(rle.split(), dtype=np.int64)
            starts = numbers[0::2] - 1
            lengths = numbers[1::2]
            flat = np.zeros(HEIGHT * WIDTH, dtype=np.uint8)
            for s, l in zip(starts, lengths):
                flat[s:s + l] = 1
            masks[i, k] = flat.reshape((HEIGHT, WIDTH), order="F")   # column-by-column order

print("images:", images.shape, "| masks:", masks.shape)

images: (11393, 128, 800) | masks: (11393, 4, 128, 800)


In [4]:
# split 85% / 15%, keeping the "has any defect" balance the same in both
has_defect = masks.reshape(len(ids), -1).any(axis=1)
train_idx, val_idx = train_test_split(np.arange(len(ids)), test_size=0.15,
                                      random_state=42, stratify=has_defect)
print("train images:", len(train_idx), "| validation images:", len(val_idx))

train images: 9684 | validation images: 1709


In [5]:
# a simple encoder-decoder CNN: shrink the image 3 times, then grow it back 3 times
model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.MaxPool2d(2),                                   # 64 x 400
    nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.MaxPool2d(2),                                   # 32 x 200
    nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.MaxPool2d(2),                                   # 16 x 100
    nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
    nn.ConvTranspose2d(128, 64, 2, stride=2),          # 32 x 200
    nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
    nn.ConvTranspose2d(64, 32, 2, stride=2),           # 64 x 400
    nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
    nn.ConvTranspose2d(32, 16, 2, stride=2),           # 128 x 800
    nn.Conv2d(16, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
    nn.Conv2d(16, 4, 1),                               # 4 outputs = one mask per defect
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()
print("model parameters:", sum(p.numel() for p in model.parameters()))

model parameters: 238260


In [6]:
best_score = -1
best_weights = None
wait = 0

for epoch in range(EPOCHS):
    # ---------- train ----------
    model.train()
    order = np.random.permutation(train_idx)
    total_loss = 0
    n_batches = 0
    for start in range(0, len(order), BATCH_SIZE):
        b = order[start:start + BATCH_SIZE]
        x = torch.tensor(images[b], dtype=torch.float32).unsqueeze(1) / 255.0
        y = torch.tensor(masks[b], dtype=torch.float32)

        # random flips (image and masks flipped together)
        if np.random.rand() < 0.5:
            x, y = x.flip(3), y.flip(3)
        if np.random.rand() < 0.5:
            x, y = x.flip(2), y.flip(2)
        x, y = x.to(device), y.to(device)

        logits = model(x)
        probs = torch.sigmoid(logits)
        inter = (probs * y).sum(dim=(2, 3))
        denom = probs.sum(dim=(2, 3)) + y.sum(dim=(2, 3))
        dice_loss = 1 - (2 * inter + 1) / (denom + 1)
        loss = bce(logits, y) + dice_loss.mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    # ---------- validate (threshold 0.5) ----------
    model.eval()
    inter_sum = np.zeros(4)
    denom_sum = np.zeros(4)
    pos_count = np.zeros(4)
    with torch.no_grad():
        for start in range(0, len(val_idx), BATCH_SIZE):
            b = val_idx[start:start + BATCH_SIZE]
            x = torch.tensor(images[b], dtype=torch.float32).unsqueeze(1) / 255.0
            y = torch.tensor(masks[b], dtype=torch.float32)
            pred = torch.sigmoid(model(x.to(device))).cpu() >= 0.5
            truth = y >= 0.5
            has_truth = truth.flatten(2).any(dim=2)          # (batch, 4)
            for k in range(4):
                keep = has_truth[:, k]
                inter_sum[k] += (pred[keep, k] & truth[keep, k]).sum().item()
                denom_sum[k] += pred[keep, k].sum().item() + truth[keep, k].sum().item()
                pos_count[k] += keep.sum().item()

    fg_dice = 2 * inter_sum / np.maximum(denom_sum, 1)
    score = fg_dice[pos_count > 0].mean()        # average over defects that exist in validation
    print(f"epoch {epoch + 1}: loss {total_loss / n_batches:.4f} | val dice {score:.4f} | per defect {fg_dice.round(3)}", flush=True)

    # keep the best weights, stop if no improvement for a while
    if score > best_score:
        best_score = score
        best_weights = copy.deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print("early stopping")
            break

model.load_state_dict(best_weights)
print("best validation dice:", round(best_score, 4))

epoch 1: loss 1.1521 | val dice 0.1113 | per defect [0.    0.    0.445 0.   ]
epoch 2: loss 0.9978 | val dice 0.2395 | per defect [0.    0.    0.456 0.502]
epoch 3: loss 0.6490 | val dice 0.1291 | per defect [0.    0.    0.516 0.   ]
epoch 4: loss 0.3393 | val dice 0.1074 | per defect [0.   0.   0.43 0.  ]
epoch 5: loss 0.2726 | val dice 0.0055 | per defect [0.    0.    0.022 0.   ]
epoch 6: loss 0.2662 | val dice 0.0001 | per defect [0. 0. 0. 0.]
epoch 7: loss 0.2604 | val dice 0.0461 | per defect [0.    0.    0.184 0.   ]
epoch 8: loss 0.2560 | val dice 0.0391 | per defect [0.    0.    0.156 0.   ]
early stopping
best validation dice: 0.2395


In [7]:
# predict the whole validation set once and keep the probabilities
model.eval()
val_probs = np.zeros((len(val_idx), 4, HEIGHT, WIDTH), dtype=np.float16)
with torch.no_grad():
    for start in range(0, len(val_idx), BATCH_SIZE):
        b = val_idx[start:start + BATCH_SIZE]
        x = torch.tensor(images[b], dtype=torch.float32).unsqueeze(1) / 255.0
        val_probs[start:start + len(b)] = torch.sigmoid(model(x.to(device))).cpu().numpy()

# for each defect, pick the threshold with the best score
# score = 70% dice on images that have the defect + 30% accuracy on images that don't
candidates = np.arange(0.10, 0.951, 0.05)
thresholds = []
for k in range(4):
    truth = masks[val_idx, k].astype(bool)
    has_truth = truth.reshape(len(truth), -1).any(axis=1)
    best_t, best_s = 0.5, -1
    for t in candidates:
        pred = val_probs[:, k] >= t
        pred_any = pred.reshape(len(pred), -1).any(axis=1)

        if has_truth.sum() > 0:
            inter = (pred[has_truth] & truth[has_truth]).sum()
            denom = pred[has_truth].sum() + truth[has_truth].sum()
            dice = 2 * inter / denom if denom > 0 else 0
        else:
            dice = 0
        if (~has_truth).sum() > 0:
            empty_acc = (~pred_any[~has_truth]).mean()
        else:
            empty_acc = 1

        s = 0.7 * dice + 0.3 * empty_acc
        if s > best_s:
            best_t, best_s = t, s
    thresholds.append(best_t)
    print(f"{defects[k]}: best threshold {best_t:.2f} (score {best_s:.4f})")

defect_a: best threshold 0.10 (score 0.3000)
defect_b: best threshold 0.10 (score 0.3000)
defect_c: best threshold 0.30 (score 0.4076)
defect_d: best threshold 0.60 (score 0.4820)


In [8]:
# test images (same order / ids as the sample submission)
test_ids = sample["image_id"].drop_duplicates().tolist()
test_dir = root + "/test"
if not os.path.isdir(test_dir):
    test_dir = root + "/test_images"

test_files = {}
for path in glob.glob(test_dir + "/**/*.*", recursive=True):
    name = os.path.splitext(os.path.basename(path))[0]
    test_files[name] = path
print("test images:", len(test_ids), "| found on disk:", sum(i in test_files for i in test_ids))

predictions = {}      # (image_id, defect) -> rle text
model.eval()
with torch.no_grad():
    for start in range(0, len(test_ids), BATCH_SIZE):
        batch_ids = test_ids[start:start + BATCH_SIZE]
        batch = np.stack([np.array(Image.open(test_files[i]).convert("L")) for i in batch_ids])
        x = torch.tensor(batch, dtype=torch.float32).unsqueeze(1) / 255.0
        probs = torch.sigmoid(model(x.to(device))).cpu().numpy()

        for n in range(len(batch_ids)):
            for k in range(4):
                mask = probs[n, k] >= thresholds[k]
                # mask -> RLE text (column by column, 1-based start + length)
                pixels = np.concatenate(([0], mask.reshape(-1, order="F").astype(np.uint8), [0]))
                changes = np.flatnonzero(pixels[1:] != pixels[:-1])
                starts = changes[0::2] + 1
                lengths = changes[1::2] - changes[0::2]
                if len(starts) == 0:
                    rle = EMPTY_VALUE            # no defect -> never leave it blank
                else:
                    rle = " ".join(f"{s} {l}" for s, l in zip(starts, lengths))
                predictions[(batch_ids[n], defects[k])] = rle
print("predictions made:", len(predictions))

test images: 3000 | found on disk: 3000
predictions made: 12000
